In [ ]:
from rampr.io import national
import pandas as pd
import numpy as np

In [ ]:
use, make, U, V, g, q= national.read_use_make()

In [ ]:
U.shape

In [ ]:
gd = g + 1 * (g==0)

In [ ]:
B = U.values[:402, :402] @ np.diag(1/gd)

In [ ]:
qd = q + 1 * (q==0)

In [ ]:
D = V.values[:402, :402] @ np.diag(1/qd)

In [ ]:
A = D @ B

In [ ]:
L = np.linalg.inv(np.eye(402) - A)

In [ ]:
L.sum(axis=0)

In [ ]:
g

In [ ]:
U.shape

In [ ]:
V.shape

In [ ]:
g = U.values[-1,:402]

In [ ]:
g.shape

In [ ]:
q = U.values[:402,-1]

In [ ]:
q.shape

In [ ]:
q

## Closed model

In [ ]:
compensation = U.values[403,:402]
wage_coefficient = compensation / gd

In [ ]:
wage_coefficient.max()

In [ ]:
wage_coefficient.mean()

In [ ]:
wage_coefficient[wage_coefficient>1.0]

In [ ]:
AC = np.zeros((403,403))

In [ ]:
AC[:402,:402] = A

In [ ]:
AC[402,:402] = wage_coefficient

In [ ]:
PCE = U.values[:402,403]
PCE.shape

In [ ]:
PCE

In [ ]:
c = PCE/PCE.sum()

In [ ]:
c

In [ ]:
AC[:402,402] = c

In [ ]:
AC[:,402]

In [ ]:
c.max()

In [ ]:
LC = np.linalg.inv(np.eye(403)-AC)

In [ ]:
MC = LC.sum(axis=0)

In [ ]:
MC

In [ ]:
A[0:4,0:4]

In [ ]:
AC[0:4, 0:4]

## Multipliers

In [ ]:
SO = L.sum(axis=0)

In [ ]:
np.median(SO)

In [ ]:
TIIO = LC.sum(axis=0)[:-1]

In [ ]:
np.median(TIIO)

In [ ]:
U.iloc[:402,[0,1]]

In [ ]:
SO[0:4]

In [ ]:
TIIO[0:4]

In [ ]:
A.shape, g.shape

In [ ]:
Z = A @ np.diag(g)

In [ ]:
Z[0:4, 0:4]

In [ ]:
Z.shape

In [ ]:
# nonzero cells
nzc = (Z==0).sum(axis=1)

In [ ]:
nzc[0:4]

In [ ]:
U.head()

In [ ]:
sectors = U.columns[0:402]

In [ ]:
sectors

In [ ]:
U.sort_values(by='Grain farming', ascending=False).head(25)

In [ ]:
U.index

In [ ]:
sectors[[12,1, 275, 250, 249, 234, 321, 23, 290]]

In [ ]:
U402 = U.values[0:402, 1]

In [ ]:
U402.shape

In [ ]:
sectors = sectors.values

In [ ]:
U402.searchsorted?

a = np.array([40, 10, 20, 30])
sorter = np.argsort(a)
sorter
result = np.searchsorted(a, 25, sorter=sorter)
result
a[sorter[result]]

In [ ]:
sidx = np.argsort(U402)

In [ ]:
U402[sidx]

In [ ]:
sidx[::-1]

In [ ]:
U402[sidx[::-1][:10]].sum() / U402.sum()

In [ ]:
U402[sidx[::-1][:20]].sum() / U402.sum()

In [ ]:
U402[sidx[::-1][:30]].sum() / U402.sum()

In [ ]:
def disaggregate_aij(aij, zeta_i, alpha):
    aij1 = aij / ((1-alpha) + alpha * zeta_i)
    aij2 = aij1 * zeta_i

    return (aij, aij1, aij2)

In [ ]:
aij, aij1, aij2 = disaggregate_aij(0.25, 1.24, .4)
aij, aij1, aij2

In [ ]:
aij1 *.6 + .4 * aij2

In [ ]:
aij, aij1, aij2 = disaggregate_aij(0.25, 0.75, .4)

In [ ]:
aij1 *.6 + .4 * aij2

In [ ]:
aij1, aij2

In [ ]:
import numpy as np


def disaggregate_sector(A, j, alpha, zeta, copy=True):
    """
    Disaggregate sector j in an input-output coefficient matrix A into
    two subsectors:
        j1 = annual grain
        j2 = perennial grain

    Parameters
    ----------
    A : array_like, shape (n, n)
        Technical coefficients matrix.
    j : int
        Index of the sector/column to split.
    alpha : float
        Share of the original sector allocated to the second subsector
        (e.g. perennial grain). Must satisfy 0 <= alpha <= 1.
    zeta : array_like, shape (n,)
        Row-specific relative intensity coefficients, where
            zeta[i] = a[i, j, 2] / a[i, j, 1]
    copy : bool, default True
        If True, work on a copy of A.

    Returns
    -------
    A_new : ndarray, shape (n + 1, n + 1)
        Expanded coefficient matrix with sector j replaced by two sectors.
        The ordering is:
            [0, ..., j-1, j1, j2, j+1, ..., n-1]
    info : dict
        Dictionary containing useful intermediate results:
            - 'a1': first split column
            - 'a2': second split column
            - 'original_column': original A[:, j]
            - 'alpha': alpha
            - 'zeta': zeta

    Notes
    -----
    This function only splits the *column* j, i.e. the input structure
    of the sector. If you also want to split the *row* j (sales/output
    distribution), that requires an additional assumption/system.

    The original row/column j is replaced by two sectors, but the row
    split here is done by simple duplication with proportional allocation:
        row_j1 = (1 - alpha) * row_j
        row_j2 = alpha * row_j
    except for the intersection block, which is filled consistently from
    the split columns.

    In many applications, you may want a more defensible row-splitting rule.
    """
    A = np.array(A, dtype=float, copy=copy)
    n, m = A.shape
    if n != m:
        raise ValueError("A must be a square matrix.")

    if not (0 <= j < n):
        raise IndexError("j is out of bounds.")

    if not (0 <= alpha <= 1):
        raise ValueError("alpha must be between 0 and 1.")

    zeta = np.asarray(zeta, dtype=float)
    if zeta.shape != (n,):
        raise ValueError(f"zeta must have shape ({n},), got {zeta.shape}.")

    if np.any(zeta < 0):
        raise ValueError("All zeta values must be nonnegative.")

    # Original column to split
    a = A[:, j]

    # Denominator for exact identity preservation
    denom = (1 - alpha) + alpha * zeta
    if np.any(denom == 0):
        raise ValueError("Encountered zero denominator in split formula.")

    # Split columns
    a1 = a / denom
    a2 = zeta * a1

    # Build expanded matrix
    A_new = np.zeros((n + 1, n + 1), dtype=float)

    # Indices in new matrix
    # old indices < j stay the same
    # old index j becomes j and j+1
    # old indices > j shift by +1
    def new_index(old_idx):
        if old_idx < j:
            return old_idx
        elif old_idx > j:
            return old_idx + 1
        else:
            raise ValueError("Original split index maps to two positions.")

    # Copy all unaffected cells
    for r in range(n):
        for c in range(n):
            if r == j or c == j:
                continue
            rr = new_index(r)
            cc = new_index(c)
            A_new[rr, cc] = A[r, c]

    # Split the target column j into j and j+1
    for r in range(n):
        rr = r if r < j else r + 1
        if r == j:
            # handled below in 2x2 block
            continue
        A_new[rr, j] = a1[r]
        A_new[rr, j + 1] = a2[r]

    # Split the target row j proportionally
    row = A[j, :]
    for c in range(n):
        if c == j:
            continue
        cc = c if c < j else c + 1
        A_new[j, cc] = (1 - alpha) * row[c]
        A_new[j + 1, cc] = alpha * row[c]

    # Fill the 2x2 intersection block consistently
    # Original self-coefficient A[j, j] is split using same rule
    a_jj = A[j, j]
    denom_j = (1 - alpha) + alpha * zeta[j]
    a1_j = a_jj / denom_j
    a2_j = zeta[j] * a1_j

    # Allocate self/input relations proportionally by output shares
    A_new[j, j] = (1 - alpha) * a1_j
    A_new[j + 1, j] = alpha * a1_j
    A_new[j, j + 1] = (1 - alpha) * a2_j
    A_new[j + 1, j + 1] = alpha * a2_j

    info = {
        "a1": a1,
        "a2": a2,
        "original_column": a,
        "alpha": alpha,
        "zeta": zeta,
    }

    return A_new, info

In [ ]:
A = np.array([
    [0.10, 0.20, 0.15],
    [0.05, 0.10, 0.20],
    [0.08, 0.12, 0.10],
])

# Split sector 1 (grain farming)
alpha = 0.25
zeta = np.array([1.2, 0.8, 1.5])

A_new, info = disaggregate_sector(A, j=1, alpha=alpha, zeta=zeta)

print("Original A:")
print(A)
print("\nExpanded A:")
print(A_new)
print("\nAnnual column:")
print(info["a1"])
print("\nPerennial column:")
print(info["a2"])

In [ ]:
A = np.array([
    [0.10, 0.20, 0.15],
    [0.05, 0.10, 0.20],
    [0.08, 0.12, 0.10],
])

# Split sector 1 (grain farming)
alpha = 0.25
zeta = np.array([1.2, 1, 1.])

A_new, info = disaggregate_sector(A, j=1, alpha=alpha, zeta=zeta)

print("Original A:")
print(A)
print("\nExpanded A:")
print(A_new)
print("\nAnnual column:")
print(info["a1"])
print("\nPerennial column:")
print(info["a2"])

In [ ]:
A = np.array([
    [0.10, 0.20, 0.15],
    [0.05, 0.10, 0.20],
    [0.08, 0.12, 0.10],
])

# Split sector 1 (grain farming)
alpha = 0.10
zeta = np.array([1.2, 1, 1.])

A_new, info = disaggregate_sector(A, j=1, alpha=alpha, zeta=zeta)

print("Original A:")
print(A)
print("\nExpanded A:")
print(A_new)
print("\nAnnual column:")
print(info["a1"])
print("\nPerennial column:")
print(info["a2"])